# Chroma DB — Query Notebook

This notebook lets you query the local `pql_docs` Chroma collection.

It is structured in four sections:

| # | Section | What it does |
|---|---|---|
| 1 | **Setup** | Connect to the DB and print basic stats |
| 2 | **Explore** | Browse chunks by type, peek at raw content |
| 3 | **Semantic Search** | Embed a natural-language question and retrieve the closest chunks |
| 4 | **Filtered Search** | Narrow results by `chunk_type`, `term_name`, or both |

> **Prerequisites**: `OPENAI_API_KEY` must be set in a `.env` file at the project root (needed for sections 3 & 4).

---
## 1 — Setup

In [1]:
import os
from pathlib import Path
from collections import Counter

import chromadb
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(Path("../.env"))

CHROMA_PATH     = Path("../data/chroma").resolve()
COLLECTION_NAME = "pql_docs"
EMBED_MODEL     = "text-embedding-3-small"   # must match what pipeline.py used

# Connect
db         = chromadb.PersistentClient(path=str(CHROMA_PATH))
collection = db.get_collection(COLLECTION_NAME)

print(f"Connected to : {CHROMA_PATH}")
print(f"Collection   : {COLLECTION_NAME}")
print(f"Total chunks : {collection.count()}")

Connected to : /Users/adrianmarino/adrianmarino19/pql-agent/data/chroma
Collection   : pql_docs
Total chunks : 943


---
## 2 — Explore the Collection

In [2]:
# --- Chunk-type breakdown ---
# Fetch ALL metadata (no embeddings, no documents — fast)
all_meta = collection.get(include=["metadatas"])["metadatas"]

type_counts = Counter(m["chunk_type"] for m in all_meta)
print("Chunk types:")
for ctype, count in sorted(type_counts.items(), key=lambda x: -x[1]):
    print(f"  {ctype:<25} {count:>4} chunks")

Chunk types:
  example                    570 chunks
  full                       188 chunks
  description_syntax         120 chunks
  concept                     65 chunks


In [3]:
# --- Peek at the first 10 chunks (raw, no embedding needed) ---
peek = collection.peek(limit=10)

for i, (chunk_id, meta, doc) in enumerate(
    zip(peek["ids"], peek["metadatas"], peek["documents"]), start=1
):
    print(f"\n{'='*65}")
    print(f"[{i}] id          : {chunk_id}")
    print(f"    title       : {meta.get('title')}")
    print(f"    term_name   : {meta.get('term_name') or '—'}")
    print(f"    chunk_type  : {meta.get('chunk_type')}")
    print(f"    tokens      : {meta.get('token_count')}")
    print(f"    url         : {meta.get('url')}")
    print(f"    preview     : {' '.join(doc.split())[:220]}…")


[1] id          : 903a5ebb6a370bd7
    title       : PQL - Process Query Language
    term_name   : —
    chunk_type  : concept
    tokens      : 1134
    url         : https://docs.celonis.com/en/pql---process-query-language.html
    preview     : pql - Process Query Language Description The Process Query Language (pql) is the analytical backbone of the Celonis platform, and empowers you to translate complex business questions into actionable process insights. pql…

[2] id          : bbc0570ebe89928a
    title       : Cheat Sheets
    term_name   : —
    chunk_type  : concept
    tokens      : 22
    url         : https://docs.celonis.com/en/cheat-sheets.html
    preview     : Cheat Sheets Description Those pql cheat sheets summarize important pql functionality. source / target : PU functions :…

[3] id          : 033d60857b6d4348
    title       : Data Model Design
    term_name   : —
    chunk_type  : concept
    tokens      : 1504
    url         : https://docs.celonis.com/en/data

In [ ]:
# --- Full chunk text (not truncated) ---
# `peek` already has the full string in `peek["documents"]`; the preview cell only slices [:220].
# Pick a row from the peek list (0 = first, 1 = second, …):
INDEX = 0
print(peek["documents"][INDEX])

# Or load any chunk by id (copy the `id` line from peek, or from query results):
# chunk_id = "903a5ebb6a370bd7"
# out = collection.get(ids=[chunk_id], include=["documents", "metadatas"])
# print(out["metadatas"][0])
# print(out["documents"][0])

In [4]:
# --- Browse chunks that have a known term_name ---
named = collection.get(
    where={"term_name": {"$ne": ""}},
    include=["metadatas"],
    limit=20,
)

print(f"Chunks with a term_name (first 20 of many):")
for m in named["metadatas"]:
    print(f"  {m['term_name']:<35} [{m['chunk_type']}]")

Chunks with a term_name (first 20 of many):
  INT                                 [concept]
  FLOAT                               [concept]
  STRING                              [concept]
  DATE                                [concept]
  NULL                                [concept]
  AVG                                 [full]
  COUNT_TABLE                         [full]
  COUNT                               [full]
  COUNT                               [full]
  FIRST                               [full]
  GLOBAL                              [full]
  LAST                                [full]
  MAX                                 [full]
  MEDIAN                              [full]
  MIN                                 [full]
  MODE                                [full]
  PRODUCT                             [full]
  QUANTILE                            [full]
  STDEV                               [full]
  STRING_AGG                          [full]


---
## 3 — Semantic Search

Ask a natural-language question. The question is embedded with OpenAI and the
closest chunks are returned by cosine similarity.

**Lower distance = more similar** (range 0 – 2 in cosine space).

In [5]:
def embed(text: str) -> list[float]:
    """Embed a single string using the same model that was used during ingestion."""
    oai = OpenAI()  # reads OPENAI_API_KEY from env automatically
    return oai.embeddings.create(model=EMBED_MODEL, input=text).data[0].embedding


def semantic_search(
    question: str,
    n_results: int = 5,
    where: dict | None = None,
) -> None:
    """
    Embed *question*, query Chroma, and pretty-print the top results.

    Parameters
    ----------
    question  : Natural-language question or PQL term you are looking for.
    n_results : How many chunks to return (default 5).
    where     : Optional Chroma metadata filter dict (see Section 4).
    """
    vector = embed(question)

    kwargs = dict(
        query_embeddings=[vector],
        n_results=n_results,
        include=["documents", "metadatas", "distances"],
    )
    if where:
        kwargs["where"] = where

    results = collection.query(**kwargs)

    print(f"Query : '{question}'")
    print(f"Filter: {where or 'none'}\n")

    for rank, (dist, meta, doc) in enumerate(
        zip(results["distances"][0], results["metadatas"][0], results["documents"][0]),
        start=1,
    ):
        print(f"{'─'*65}")
        print(f"#{rank}  distance={dist:.4f}  |  {meta['chunk_type']}")
        print(f"    title     : {meta.get('title')}")
        print(f"    term_name : {meta.get('term_name') or '—'}")
        print(f"    url       : {meta.get('url')}")
        print(f"    text      : {' '.join(doc.split())[:350]}…")
        print()

In [6]:
# ✏️  Change this question to whatever you want to look up
semantic_search(
    question="How do I calculate the duration between two events in a process?",
    n_results=5,
)

Query : 'How do I calculate the duration between two events in a process?'
Filter: none

─────────────────────────────────────────────────────────────────
#1  distance=0.5380  |  concept
    title     : Machine Utilization in Production
    term_name : —
    url       : https://docs.celonis.com/en/machine-utilization-in-production.html
    text      : Machine Utilization in Production Description This example shows how to calculate the throughput time of a machine with pql. In production use cases machine throughput time is often a number of interest. Calculating this number is not straight forward as the event log is usually built around the products which go through the assembly line and not t…

─────────────────────────────────────────────────────────────────
#2  distance=0.5623  |  concept
    title     : Throughput Times
    term_name : —
    url       : https://docs.celonis.com/en/throughput-times.html
    text      : Throughput Times Description These examples show various ways 

In [7]:
semantic_search(
    question="How do I count distinct values per case?",
    n_results=5,
)

Query : 'How do I count distinct values per case?'
Filter: none

─────────────────────────────────────────────────────────────────
#1  distance=0.3566  |  full
    title     : COUNT DISTINCT
    term_name : COUNT
    url       : https://docs.celonis.com/en/count-distinct.html
    text      : count distinct Description This function calculates the number of distinct elements per group. count distinct can be applied on any data type. Syntax count ( distinct table.column ) null handling null values are not counted. If all the values of a group are null, the result for this group is 0. count distinct over multiple columns unique id can be …

─────────────────────────────────────────────────────────────────
#2  distance=0.3908  |  example
    title     : PU_COUNT_DISTINCT
    term_name : PU_COUNT_DISTINCT
    url       : https://docs.celonis.com/en/pu_count_distinct.html
    text      : Function: pu count distinct Syntax: pu count distinct ( target_table, source_table.column [, filter_expre

In [8]:
semantic_search(
    question="MATCH_PROCESS_REGEX",
    n_results=5,
)

Query : 'MATCH_PROCESS_REGEX'
Filter: none

─────────────────────────────────────────────────────────────────
#1  distance=0.4086  |  description_syntax
    title     : MATCH_PROCESS_REGEX
    term_name : MATCH_PROCESS_REGEX
    url       : https://docs.celonis.com/en/match_process_regex.html
    text      : match process regex Description Filters the variants based on a regular expression defined over the activities. match process regex matches the variants of a process based on a regular expression. The regular expression defines a pattern over the activities of the variant. If the regular expression contains an activity name that does not exist, the…

─────────────────────────────────────────────────────────────────
#2  distance=0.4104  |  example
    title     : MATCH_PROCESS_REGEX
    term_name : MATCH_PROCESS_REGEX
    url       : https://docs.celonis.com/en/match_process_regex.html
    text      : Function: match process regex Syntax: match process regex ( activity_table.string_

---
## 4 — Filtered Search

Combine semantic search with a **metadata filter** using the `where` parameter.

Available filter fields and their values in this collection:

| Field | Example values |
|---|---|
| `chunk_type` | `"concept"`, `"full"`, `"description_syntax"`, `"example"` |
| `term_name` | `"MATCH_PROCESS_REGEX"`, `"PU_COUNT_DISTINCT"`, `""` (empty = no term) |
| `title` | Any page title string |

Chroma filter operators: `$eq`, `$ne`, `$in`, `$nin`, `$and`, `$or`

In [9]:
# Only look in 'example' chunks — the most concrete/code-like chunks
semantic_search(
    question="How do I filter cases that went through a specific activity?",
    n_results=5,
    where={"chunk_type": "example"},
)

Query : 'How do I filter cases that went through a specific activity?'
Filter: {'chunk_type': 'example'}

─────────────────────────────────────────────────────────────────
#1  distance=0.3647  |  example
    title     : SOURCE - TARGET
    term_name : —
    url       : https://docs.celonis.com/en/source---target.html
    text      : Function: source - target [6] In this example, we filter based on the temporary edge table. The filter only keeps edges where the source activity equals 'A'. Since case 2 does not contain a source activity 'A', it is filtered out completely by the filter. Case 1 however contains an edge with an 'A' as the source activity (edge 'A' to 'B'), which is…

─────────────────────────────────────────────────────────────────
#2  distance=0.4066  |  example
    title     : MATCH_ACTIVITIES
    term_name : MATCH_ACTIVITIES
    url       : https://docs.celonis.com/en/match_activities.html
    text      : Function: match activities Syntax: match activities([ activity_tab

In [10]:
# Only look in chunks that have a named PQL term (term_name is not empty)
semantic_search(
    question="string manipulation functions",
    n_results=5,
    where={"term_name": {"$ne": ""}},
)

Query : 'string manipulation functions'
Filter: {'term_name': {'$ne': ''}}

─────────────────────────────────────────────────────────────────
#1  distance=0.6010  |  concept
    title     : STRING
    term_name : STRING
    url       : https://docs.celonis.com/en/string.html
    text      : string Description The string type handles sequences of characters, typically text. There are various String Functions available for processing string input. When entering string constants manually, surround them with single quotes ( '...' ). Single quotes within a string constant have to be escaped with a backslash ( \' ). A literal backslash with…

─────────────────────────────────────────────────────────────────
#2  distance=0.6208  |  example
    title     : CONCAT
    term_name : CONCAT
    url       : https://docs.celonis.com/en/concat.html
    text      : Function: concat Syntax: concat ( table.column1, ..., table.columnN ) [4] float value conversions Query Column1 '' || "Table1"."Value" Inpu

In [11]:
# Combine two conditions with $and:
# - chunk_type must be 'example' or 'full'
# - term_name must be set
semantic_search(
    question="How to use PU functions to aggregate from a source table?",
    n_results=5,
    where={
        "$and": [
            {"chunk_type": {"$in": ["example", "full"]}},
            {"term_name": {"$ne": ""}},
        ]
    },
)

Query : 'How to use PU functions to aggregate from a source table?'
Filter: {'$and': [{'chunk_type': {'$in': ['example', 'full']}}, {'term_name': {'$ne': ''}}]}

─────────────────────────────────────────────────────────────────
#1  distance=0.3029  |  example
    title     : PU_SUM
    term_name : PU_SUM
    url       : https://docs.celonis.com/en/pu_sum.html
    text      : Function: pu sum Syntax: pu sum ( target_table, source_table.column [, filter_expression] ) [3] PU-functions can be used inside another aggregation function . In this example, the maximum value of all accumulated case table values for each company code is calculated: Query Column1 max ( pu sum ( "companyDetail" , "caseTable"."value" ) ) Input Outpu…

─────────────────────────────────────────────────────────────────
#2  distance=0.3125  |  full
    title     : Pull Up Aggregation
    term_name : PU_X
    url       : https://docs.celonis.com/en/pull-up-aggregation.html
    text      : Pull Up Aggregation Description 

In [12]:
# ✏️  Free cell — try your own query and filter
MY_QUESTION = "How do I calculate a running total?"
MY_FILTER   = None   # set to e.g. {"chunk_type": "example"} or leave None

semantic_search(
    question=MY_QUESTION,
    n_results=5,
    where=MY_FILTER,
)

Query : 'How do I calculate a running total?'
Filter: none

─────────────────────────────────────────────────────────────────
#1  distance=0.3816  |  full
    title     : RUNNING_TOTAL
    term_name : RUNNING_TOTAL
    url       : https://docs.celonis.com/en/running_total.html
    text      : running total Description running total sums up all entries of a given column and returns all intermediate sums. It can be applied to int or float columns. running total sums up all entries of a given column, but instead of aggregating all values into a single total result, it returns all intermediate sums. running total starts the summation at the…

─────────────────────────────────────────────────────────────────
#2  distance=0.4531  |  example
    title     : RUNNING_SUM
    term_name : RUNNING_SUM
    url       : https://docs.celonis.com/en/running_sum.html
    text      : Function: running sum Syntax: running sum ( column [, order BY ( sort_column [sorting], ... ) [5] Simple example for Runni